In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from torch import nn
import torch.optim as optim

# Task 1: Chuẩn bị dữ liệu

## Tải và tiền xử lý dữ liệu

In [2]:
def load_conllu(file_path):
    rows = []

    with open(file_path, "r", encoding="utf-8") as f:
        sent_id = 0
        for line in f:
            line = line.strip()

            # Dòng trống → sang câu mới
            if line == "":
                sent_id += 1
                continue

            # Bỏ comment
            if line.startswith("#"):
                continue

            parts = line.split("\t")
            if len(parts) != 10:
                continue

            word = parts[1]
            upos = parts[3]

            if upos == '_' or upos == '':
                continue

            rows.append([sent_id, word, upos])

    # Đưa vào pandas DataFrame
    df = pd.DataFrame(rows, columns=["sentence_id", "word", "upos"])

    # Gom theo câu → list các list[(word, upos)]
    sentences = (
        df.groupby("sentence_id")
          .apply(lambda g: list(zip(g["word"], g["upos"])))
          .tolist()
    )
    return sentences, df


In [3]:
sentences_train, df_train = load_conllu("..\\data\\UD_English-EWT\\en_ewt-ud-train.conllu")
df_train.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_14612\2014729529.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g["word"], g["upos"])))


,sentence_id,word,upos
0,0,Al,PROPN
1,0,-,PUNCT
2,0,Zaman,PROPN
3,0,:,PUNCT
4,0,American,ADJ


In [4]:
sentences_dev, df_dev = load_conllu("..\\data\\UD_English-EWT\\en_ewt-ud-dev.conllu")
df_dev.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_14612\2014729529.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g["word"], g["upos"])))


,sentence_id,word,upos
0,0,From,ADP
1,0,the,DET
2,0,AP,PROPN
3,0,comes,VERB
4,0,this,DET


## Xây dựng từ điển

In [5]:
def build_vocab(sentences):
    word_to_ix = {"<PAD>": 0, "<UNK>": 1}

    tag_to_ix = {}

    for sentence in sentences:
        for word, tag in sentence:
            # Thêm từ vào word_to_ix 
            if word not in word_to_ix:
                word_to_ix[word] = len(word_to_ix)

            # Thêm tag vào tag_to_ix 
            if tag not in tag_to_ix:
                tag_to_ix[tag] = len(tag_to_ix)

    return word_to_ix, tag_to_ix


In [6]:
word_to_ix, tag_to_ix = build_vocab(sentences_train)

print("Kích thước word_to_ix:", len(word_to_ix))
print("Kích thước tag_to_ix:", len(tag_to_ix))

Kích thước word_to_ix: 19675
Kích thước tag_to_ix: 17


# Task 2: Tạo PyTorch Dataset và DataLoader

## 2.1: POSDataset

In [7]:
class POSDataset(Dataset):
    def __init__(self, sentences, word_to_ix, tag_to_ix):
        """
        sentences: danh sách các câu [ [('From','ADP'), ('the','DET'), ...], ... ]
        word_to_ix, tag_to_ix: từ điển từ và tag
        """
        self.sentences = sentences
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]

        # Chuyển từ và tag sang index
        word_indices = torch.tensor(
            [self.word_to_ix.get(word, self.word_to_ix["<UNK>"]) for word, tag in sentence],
            dtype=torch.long
        )

        tag_indices = torch.tensor(
            [self.tag_to_ix[tag] for word, tag in sentence],
            dtype=torch.long
        )

        return word_indices, tag_indices


## Padding

In [8]:
def collate_fn(batch):
    word_seqs = [item[0] for item in batch]
    tag_seqs = [item[1] for item in batch]

    # Pad về cùng độ dài bằng PAD=0 (mặc định)
    word_padded = pad_sequence(word_seqs, batch_first=True, padding_value=0)
    tag_padded = pad_sequence(tag_seqs, batch_first=True, padding_value=0)

    return word_padded, tag_padded


## DataLoader

In [9]:
train_dataset = POSDataset(sentences_train, word_to_ix, tag_to_ix)
dev_dataset   = POSDataset(sentences_dev,  word_to_ix, tag_to_ix)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)


# Task 3: Xây dựng Mô hình RNN

In [10]:
class SimpleRNNForTokenClassification(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_tags):
        super().__init__()

        # Lớp Embedding:
        # Chuyển từ ID → vector embedding (batch, seq_len, embedding_dim)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Lớp RNN:
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True  
        )

        # Lớp Linear:
        # Hidden state (hidden_dim) → số lượng nhãn UPOS
        self.linear = nn.Linear(hidden_dim, num_tags)

    def forward(self, sentences):

        # Embedding: (batch, seq_len) → (batch, seq_len, embedding_dim)
        embeds = self.embedding(sentences)

        # RNN xử lý từng token
        # rnn_out: (batch, seq_len, hidden_dim)
        rnn_out, _ = self.rnn(embeds)

        # Linear dự đoán nhãn cho từng token
        # tag_scores: (batch, seq_len, num_tags)
        tag_scores = self.linear(rnn_out)

        return tag_scores


# Task 4: Huấn luyện Mô hình

## Khởi tạo mô hình

In [11]:

# Các tham số cho mô hình
vocab_size = len(word_to_ix)       
num_tags = len(tag_to_ix)          
PAD_TAG_ID = 0                    

model = SimpleRNNForTokenClassification(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=256,
    num_tags=num_tags
)

optimizer = optim.Adam(model.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_TAG_ID)

## Vòng lặp huấn luyện

In [12]:
def train_model(model, train_loader, dev_loader, optimizer, criterion, num_epochs=5, device="cpu"):
    model.to(device)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for sentences, tags in train_loader:
            sentences = sentences.to(device)  
            tags = tags.to(device)            
            # (1) Xóa gradient cũ
            optimizer.zero_grad()

            # (2) Forward pass
            tag_scores = model(sentences)     

            # (3) Tính loss
            loss = criterion(
                tag_scores.view(-1, tag_scores.size(-1)),
                tags.view(-1)
            )

            # (4) Lan truyền ngược
            loss.backward()
            # (5) Cập nhật trọng số
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_loss:.4f}")


        dev_loss = evaluate_loss(model, dev_loader, criterion, device)
        print(f"             Dev Loss: {dev_loss:.4f}\n")


def evaluate_loss(model, data_loader, criterion, device="cpu"):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for sentences, tags in data_loader:
            sentences = sentences.to(device)
            tags = tags.to(device)

            tag_scores = model(sentences)

            loss = criterion(
                tag_scores.view(-1, tag_scores.size(-1)),
                tags.view(-1)
            )
            total_loss += loss.item()

    return total_loss / len(data_loader)


# Task 5: Đánh giá Mô hình

## Viết hàm evaluate

In [13]:
def evaluate(model, data_loader, tag_pad_id, device="cpu"):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for sentences, tags in data_loader:
            sentences = sentences.to(device)

            tags = tags.to(device)

            outputs = model(sentences)   

            predictions = torch.argmax(outputs, dim=-1)  

            mask = (tags != tag_pad_id)  

            correct += (predictions[mask] == tags[mask]).sum().item()
            total += mask.sum().item()

    return correct / total if total > 0 else 0.0


## Kết quả

In [14]:
def train_and_evaluate(model, train_loader, dev_loader,
                       optimizer, criterion, tag_pad_id,
                       num_epochs=5, device="cpu"):

    model.to(device)
    best_dev_acc = 0.0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for sentences, tags in train_loader:
            sentences = sentences.to(device)
            tags = tags.to(device)

            optimizer.zero_grad()
            outputs = model(sentences)

            loss = criterion(outputs.view(-1, outputs.size(-1)),
                             tags.view(-1))

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        train_acc = evaluate(model, train_loader, tag_pad_id, device)
        dev_acc = evaluate(model, dev_loader, tag_pad_id, device)

        print(f"Epoch {epoch+1} | Loss = {avg_loss:.4f} | Train = {train_acc:.4f} | Dev = {dev_acc:.4f}")


        # --- Save best model ---
        if dev_acc > best_dev_acc:
            best_dev_acc = dev_acc
            torch.save(model.state_dict(), "best_model.pt")

In [15]:
train_and_evaluate(
    model=model,
    train_loader=train_loader,
    dev_loader=dev_loader,
    optimizer=optimizer,
    criterion=criterion,
    tag_pad_id=PAD_TAG_ID,
    num_epochs=5,
    device="cpu"
)

Epoch 1 | Loss = 0.8771 | Train = 0.8249 | Dev = 0.7921
Epoch 2 | Loss = 0.4740 | Train = 0.8823 | Dev = 0.8461
Epoch 2 | Loss = 0.4740 | Train = 0.8823 | Dev = 0.8461
Epoch 3 | Loss = 0.3462 | Train = 0.9111 | Dev = 0.8620
Epoch 3 | Loss = 0.3462 | Train = 0.9111 | Dev = 0.8620
Epoch 4 | Loss = 0.2687 | Train = 0.9316 | Dev = 0.8734
Epoch 4 | Loss = 0.2687 | Train = 0.9316 | Dev = 0.8734
Epoch 5 | Loss = 0.2127 | Train = 0.9470 | Dev = 0.8846
Epoch 5 | Loss = 0.2127 | Train = 0.9470 | Dev = 0.8846


In [16]:
# Đánh giá độ chính xác cuối cùng trên tập dev
final_dev_accuracy = evaluate(model, dev_loader, PAD_TAG_ID, device="cpu")
print(f"Độ chính xác cuối cùng trên tập Dev: {final_dev_accuracy:.4f} ({final_dev_accuracy*100:.2f}%)")

Độ chính xác cuối cùng trên tập Dev: 0.8846 (88.46%)


In [24]:
sentences_test, df_test = load_conllu("..\\data\\UD_English-EWT\\en_ewt-ud-test.conllu")
test_dataset   = POSDataset(sentences_test,  word_to_ix, tag_to_ix)

test_loader = DataLoader(test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
    )

test_accuracy = evaluate(model, test_loader, PAD_TAG_ID, device="cpu")

print(f"KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")


C:\Users\Admin\AppData\Local\Temp\ipykernel_14612\2014729529.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g["word"], g["upos"])))


KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST
Test Accuracy: 0.8815 (88.15%)


## Dự đoán câu mới

In [25]:
ix_to_tag = {idx: tag for tag, idx in tag_to_ix.items()}
def predict_sentence(sentence, model, word_to_ix, ix_to_tag, device="cpu"):
    model.eval()

    words = sentence.split()
    ids = []

    for w in words:
        ids.append(word_to_ix.get(w.lower(), word_to_ix["<UNK>"]))

    # (1, seq_len)
    tensor = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor)
        predicted = torch.argmax(outputs, dim=-1).squeeze(0)

    # Map lại sang nhãn UPOS
    pred_tags = [ix_to_tag[int(p)] for p in predicted]

    return list(zip(words, pred_tags))


In [ ]:
test_sentence = "The quick brown fox jumps over the lazy dog"

predictions = predict_sentence(
    sentence=test_sentence,
    model=model,
    word_to_ix=word_to_ix,
    ix_to_tag=ix_to_tag,
    device="cpu"
)

print(f"{'Từ':<15} {'POS Tag':<10}")
for word, tag in predictions:
    print(f"{word:<15} {tag:<10}")

Từ              POS Tag   
The             DET       
quick           ADJ       
brown           ADJ       
fox             NOUN      
jumps           VERB      
over            ADV       
the             DET       
lazy            ADJ       
dog             NOUN      
